# autoresearch-xla: Resumable 1B-Parameter FSDP Pretraining (Kaggle TPU v5e-8 & Colab GPU/TPU)

Train a **~1.036B parameter LLM from scratch** in pure JAX/XLA with:
- **True 1D FSDP Sharding**: Shards all 1B weights & NorMuon/AdamW states across all 8 TPU chips on Kaggle `v5e-8` (~1.15 GB/chip static HBM out of 16 GB/chip) or across multi-GPU setups.
- **Fast `lax.scan` Compilation**: Compiles 20 transformer layers in ~20s with O(1) activation depth.
- **Disk-Bounded ClimbMix Streaming**: Streams shards from `karpathy/climbmix-400b-shuffle` using a rolling ~200 MB `/tmp` cache + exact resumable `DataCursor` (never fills Kaggle's 20 GB disk!).
- **WSD Schedule (Warmup → Stable → Decay)**: Run as many 9h Kaggle / 12h Colab sessions as you want in `phase="stable"`, then switch to `phase="decay"` for the final cooldown!
- **Topology-Agnostic Auto-Resume**: Syncs checkpoints + tokenizer + cursor to **HuggingFace Hub** (recommended for cross-platform Kaggle ↔ Colab), **Google Drive**, or **Kaggle Datasets**.


## 1. Environment Setup (Colab / Kaggle Auto-Detection)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV_NAME = "colab" if IN_COLAB else ("kaggle" if IN_KAGGLE else "local")
print(f"Environment: {ENV_NAME}")

REPO_URL = "https://github.com/OttoApocalypse69/autoresearch-xla.git"
MOUNT_GOOGLE_DRIVE = True  # Only used on Colab

if Path("scale/pretrain.py").exists():
    REPO_DIR = Path.cwd()
else:
    target = Path("/content/autoresearch-xla" if IN_COLAB else ("/kaggle/working/autoresearch-xla" if IN_KAGGLE else "./autoresearch-xla"))
    if target.exists():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(target)], check=True)
    os.chdir(target)
    REPO_DIR = target

DRIVE_DIR = None
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = "/content/drive/MyDrive/autoresearch-1b-ckpt"
    except Exception as exc:
        print(f"Skipping Google Drive mount: {exc}")

# Install lightweight non-JAX deps ONLY (never clobber preinstalled Colab/Kaggle JAX/libtpu!)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tiktoken", "pyarrow", "requests", "tokenizers", "huggingface-hub", "safetensors", "matplotlib"],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rustbpe"], check=False)
print(f"Repo ready at {REPO_DIR}")

## 2. Inspect Hardware & FSDP Memory Plan (TPU-Lock-Safe Subprocess)

Probes your attached GPU/TPU in an isolated subprocess so the Jupyter kernel never holds Cloud TPU's `/tmp/libtpu_lockfile`.


In [ ]:
import subprocess, sys

PRESET = "1b"  # "1b" (~1.036B SwiGLU) | "1b-relu2" (~1.015B ReLU^2) | "350m" | "100m" | "smoke"

code = f'''
import functools, jax
from prepare import detect_hardware
from scale.config import get_model_preset
from scale.fsdp import create_fsdp_mesh, build_tree_shardings, summarize_sharding
from scale.model import init_params, count_params
from scale.optimizer import init_opt_state

hw = detect_hardware()
spec = get_model_preset({PRESET!r})
mesh = create_fsdp_mesh()
abs_p = jax.eval_shape(functools.partial(init_params, spec=spec), jax.random.PRNGKey(0))
abs_o = jax.eval_shape(init_opt_state, abs_p)
p_sh = build_tree_shardings(abs_p, mesh, use_fsdp=True)
o_sh = build_tree_shardings(abs_o, mesh, use_fsdp=True)
p_total = count_params(abs_p)["total"]
p_mb = sum(a.size * a.dtype.itemsize for a in jax.tree_util.tree_leaves(abs_p)) / 1024**2
o_mb = sum(a.size * a.dtype.itemsize for a in jax.tree_util.tree_leaves(abs_o)) / 1024**2
print(f"JAX {{jax.__version__}} | {{hw.device_kind}} x{{hw.num_devices}} ({{hw.platform}}, {{hw.compute_dtype}}, peak={{hw.peak_tflops_total:.1f}} TFLOPS)")
print(f"Preset {PRESET!r}: {{p_total/1e6:.2f}}M params ({{spec.n_layer}}L x {{spec.n_embd}}D, {{spec.activation}}, GQA {{spec.n_head}}:{{spec.n_kv_head}})")
print(f"Static state (fp32): params={{p_mb:.0f}} MB + opt={{o_mb:.0f}} MB = {{p_mb+o_mb:.0f}} MB total -> ~{{(p_mb+o_mb)/hw.num_devices:.0f}} MB/chip with FSDP!")
'''
res = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Hardware probe failed.")

## 3. Configure & Launch Resumable 1B FSDP Pretraining

- **Recommended persistence (`SYNC_BACKEND = "hf"`)**: Add `HF_TOKEN` to Kaggle Secrets / Colab Secrets and set `REMOTE_TARGET = "your-hf-username/my-1b-base"`. Creates a **private** repo automatically and lets you resume seamlessly across Kaggle TPU `v5e-8` sessions and Colab runtimes!
- **Colab Google Drive persistence**: Set `SYNC_BACKEND = "local"` and `REMOTE_TARGET = DRIVE_DIR`.
- **WSD multi-session workflow**:
  1. Run as many 8.5h sessions as you want with `PHASE = "stable"` (`RESUME = True` picks up exact step, optimizer state, tokenizer, and ClimbMix shard cursor automatically).
  2. When ready to finalize the model (e.g. after 15B–20B tokens), switch to `PHASE = "decay"` (`DECAY_STEPS = 4000`, ~2B cooldown tokens) for the final `val_bpb` drop!


In [ ]:
import os
import subprocess
import sys

# ---- User Configuration ----
PRESET = "1b"                     # "1b" | "1b-relu2" | "350m" | "100m" | "smoke"
ARCH_CONFIG = None                # Optional path to JSON overrides from autoresearch winner
DATASET = "climbmix"              # "climbmix" | "tinystories" | "synthetic"
TOTAL_TOKENS = 20_000_000_000     # 20B tokens (Chinchilla-optimal for 1B)

# Batching (auto-fits Kaggle TPU v5e-8 @ 4 seq/chip x 8 chips = 32 seqs/micro; reduce per_device_batch if single 16GB chip)
TOTAL_BATCH_SIZE = 524_288        # 512K tokens/optimizer step
PER_DEVICE_BATCH = 4              # 4 on TPU v5e-8 / v6e-1 / A100 / H100; use 2 on single 16GB v5e-1/T4
OPT_DTYPE = "float32"             # "float32" (default on v5e-8/G4/A100/v6e-1) or "bfloat16" (for tight 16GB single-chip)
REMAT_POLICY = "dots_with_no_batch_dims"  # "none" (max MFU on G4 96GB / H100 / v5e-8!) | "dots_with_no_batch_dims" | "full"
ATTN_IMPL = "auto"                # "auto" (Pallas Splash on TPU, cuDNN Flash on Ampere+/Blackwell GPU, XLA on T4)
LOSS_CHUNK_SIZE = 2048            # Chunked linear CE (cuts logits HBM ~16x; 0 disables)

# WSD Schedule
PHASE = "stable"                  # "stable" for multi-session runs, then "decay" for final ~10% cooldown!
WARMUP_STEPS = 500
DECAY_STEPS = 4000

# Session Safety & Checkpointing
MAX_HOURS = 8.5                   # Graceful save & sync before Kaggle's 9h TPU limit
MAX_STEPS = None                  # Set e.g. 50 for a quick sanity run; None = run until MAX_HOURS or TOTAL_TOKENS
SAVE_EVERY = 500
EVAL_EVERY = 250
SAMPLE_EVERY = 500
LOG_EVERY = 10

# Remote Checkpoint Sync ("hf" | "local" | "kaggle" | "none")
SYNC_BACKEND = "local" if DRIVE_DIR else "none"
REMOTE_TARGET = DRIVE_DIR         # e.g. "your-username/my-1b-base" for SYNC_BACKEND="hf"
CHECKPOINT_DIR = "/kaggle/working/ckpt_1b" if IN_KAGGLE else "./checkpoints/1b"
# ----------------------------

cmd = [
    sys.executable, "-m", "scale.pretrain",
    "--preset", PRESET,
    "--dataset", DATASET,
    "--total-tokens", str(TOTAL_TOKENS),
    "--total-batch-size", str(TOTAL_BATCH_SIZE),
    "--per-device-batch", str(PER_DEVICE_BATCH),
    "--opt-dtype", OPT_DTYPE,
    "--remat-policy", REMAT_POLICY,
    "--attn-impl", ATTN_IMPL,
    "--loss-chunk-size", str(LOSS_CHUNK_SIZE),
    "--phase", PHASE,
    "--warmup-steps", str(WARMUP_STEPS),
    "--decay-steps", str(DECAY_STEPS),
    "--max-hours", str(MAX_HOURS),
    "--save-every", str(SAVE_EVERY),
    "--eval-every", str(EVAL_EVERY),
    "--sample-every", str(SAMPLE_EVERY),
    "--log-every", str(LOG_EVERY),
    "--checkpoint-dir", CHECKPOINT_DIR,
    "--sync-backend", SYNC_BACKEND,
]
if ARCH_CONFIG:
    cmd += ["--arch-config", ARCH_CONFIG]
if MAX_STEPS is not None:
    cmd += ["--max-steps", str(MAX_STEPS)]
if REMOTE_TARGET:
    cmd += ["--remote-target", str(REMOTE_TARGET)]

# Pass HF_TOKEN from Colab/Kaggle Secrets if present
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
if "HF_TOKEN" not in env:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            env["HF_TOKEN"] = tok
    except Exception:
        pass
if "HF_TOKEN" not in env:
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            env["HF_TOKEN"] = tok
    except Exception:
        pass

print("Launching:", " ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Pretraining exited with code {rc}")

## 4. Plot Training Loss & `val_bpb` Curve


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

metrics_file = Path(CHECKPOINT_DIR) / "metrics.jsonl"
if metrics_file.exists():
    rows = [json.loads(line) for line in metrics_file.read_text().splitlines() if line.strip()]
    train_rows = [r for r in rows if "loss" in r]
    eval_rows = [r for r in rows if "val_bpb" in r]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=130)
    if train_rows:
        axes[0].plot([r["tokens_seen"] / 1e9 for r in train_rows], [r["loss"] for r in train_rows], color="#38bdf8")
        axes[0].set_title("Training Loss vs Tokens (B)")
        axes[0].set_xlabel("Tokens Seen (Billions)")
        axes[0].set_ylabel("Cross-Entropy Loss (nats)")
        axes[0].grid(alpha=0.3)
    if eval_rows:
        axes[1].plot([r["tokens_seen"] / 1e9 for r in eval_rows], [r["val_bpb"] for r in eval_rows], marker="o", color="#22c55e")
        axes[1].set_title("Validation val_bpb vs Tokens (B)")
        axes[1].set_xlabel("Tokens Seen (Billions)")
        axes[1].set_ylabel("val_bpb (lower is better)")
        axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No metrics.jsonl yet — run Cell 3 first.")